In [19]:
import json
from pandas import json_normalize
import pandas as pd

def flatten_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
        
    if isinstance(data, list):
        df = json_normalize(data)
    else:
        df = json_normalize([data])
        
    
    def extract_servizi(row):
        servizi = row.get('servizi') or []
        result = {}
        
        for i, servizio in enumerate(servizi):
            if not isinstance(servizio, dict):
                continue
            servizio_type = servizio.get('servizio_energetico', 'placeholder')
            prefix = f"servizio_{servizio_type}"
            
            result[f'{prefix}_epren'] = servizio.get('epren', 0) or 0
            result[f'{prefix}_epnren'] = servizio.get('epnren', 0) or 0
            result[f'{prefix}_efficienza'] = servizio.get('efficienza_media_stagionale', 0) or 0
            imp_sim = servizio.get('impianto_simulato') or ''
            result[f'{prefix}_simulato'] = 1 if 'SIMULATO' in imp_sim else 0
            
        return pd.Series(result)
    
    servizi_features = df.apply(extract_servizi, axis=1)
    df = df.drop('servizi', axis=1).join(servizi_features)
    
    return df



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

json_file = '../data/raw/AN2023.json'


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None) 



df = pd.read_json(json_file)
df = flatten_data(json_file)

df = df.drop(columns=['software_utilizzato', 'dpr412', 'motivazione', 'altra_motivazione', 'inizio_validita', 'fine_validita', 'data_sopralluogo', 'piano', 'provincia', 'comune', 'codice_istat_comune', 'oggetto_attestato', 'cap', 'proprieta_edificio'])

# Clean 'informazioni_miglioramento'
# some entries have "Data sopralluogo: 2023/01/15" or "Data del sopralluogo: 2023-01-15" which i want to remove
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].fillna('') # will use fillna on entire df later
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].astype(str)
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].str.replace(r'Data sopralluogo:\s*[\d-]{1,4}/[\d-]{1,4}/[\d-]{2,4}', '', regex=True)
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].str.replace(r'Data del sopralluogo:\s*[\d-]{1,4}/[\d-]{1,4}/[\d-]{2,4}', '', regex=True)

df['informazioni_miglioramento'] = df['informazioni_miglioramento'].str.replace(r',?\s*$', '', regex=True).str.replace(r'\s{2,}', ' ', regex=True).str.strip()
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].str.replace(r'\n', ' ')

df['informazioni_miglioramento'] = df['informazioni_miglioramento'].replace(r'^[\s\.\-–—_]+$', '', regex=True)
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].replace(r'(?i)^(n/?a|na|none|nessuno)$', '', regex=True)
# final trim
df['informazioni_miglioramento'] = df['informazioni_miglioramento'].str.strip()

# print(df.columns)
# print('\n', df['servizi']) to clean

print(df['informazioni_miglioramento'].astype(str))
